# **Complete Guide to RAG (Retrieval-Augmented Generation) & Rerankers**
## Using LangChain + Open-Source HuggingFace Models

---

**Purpose:** End-to-end hands-on guide for understanding RAG pipelines and Reranking techniques  
**Requirements:** Python 3.8+, No API keys needed — 100% open-source models  
**Estimated Runtime:** ~15 minutes on CPU

---

### What You Will Learn

| # | Topic | Description |
|---|-------|-------------|
| 1 | **Setup** | Install all packages |
| 2 | **Understanding RAG** | What is RAG and why it matters |
| 3 | **Document Loading** | Load and prepare documents |
| 4 | **Text Chunking** | Split documents into meaningful chunks |
| 5 | **Embeddings** | Convert text to numerical vectors |
| 6 | **Vector Store** | Store and index embeddings in memory |
| 7 | **Retrieval** | Find relevant chunks for a query |
| 8 | **Chains & RunnablePassthrough** | How LangChain connects components |
| 9 | **RAG Pipeline** | End-to-end question answering |
| 10 | **Rerankers** | Improve retrieval with FlashRank, Cross-Encoders & BM25 |
| 11 | **Full Pipeline** | Complete RAG + Reranking system |

---
## **PART 1: Understanding RAG**
---

### What is RAG?

**Retrieval-Augmented Generation (RAG)** is a technique that enhances LLMs by providing them with relevant context from external documents before generating a response.

### Why RAG?

| Problem with plain LLMs | How RAG solves it |
|---|---|
| Knowledge cutoff (outdated info) | Retrieves fresh documents |
| Hallucination (makes up facts) | Grounds answers in real data |
| No domain-specific knowledge | Injects your custom documents |
| Context window limits | Retrieves only relevant chunks |

### The RAG Pipeline (Step by Step)

```
┌─────────────────────────────────────────────────────────────────────┐
│                        INDEXING PHASE (Offline)                     │
│                                                                     │
│  Documents ──► Chunking ──► Embedding ──► Vector Store (FAISS)     │
│  "AI is..."    [chunk1]     [0.2, 0.8]    Index: {id→vector}       │
│                [chunk2]     [0.5, 0.3]                              │
│                [chunk3]     [0.1, 0.9]                              │
└─────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────┐
│                       RETRIEVAL PHASE (Online)                      │
│                                                                     │
│  User Query ──► Embed Query ──► Similarity Search ──► Top-K Chunks │
│  "What is DL?"  [0.4, 0.7]    cosine similarity      [chunk2,3]   │
└─────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────┐
│                       GENERATION PHASE (Online)                     │
│                                                                     │
│  Prompt Template + Retrieved Chunks + Query ──► LLM ──► Answer     │
│  "Context: {chunks}                                                 │
│   Question: {query}                                                 │
│   Answer: ..."                                                      │
└─────────────────────────────────────────────────────────────────────┘
```

---
## **PART 2: Setup & Installation**
---

We install **all packages upfront** so the rest of the notebook runs smoothly.  
All models are **open-source** from HuggingFace — no API keys required!

In [1]:
# ============================================================
# INSTALL ALL REQUIRED PACKAGES
# ============================================================
# Run this cell FIRST — it installs everything you need.
# This may take 2-3 minutes on first run.
#
# Tested with: langchain>=1.0, langchain-core>=1.0,
#   langchain-classic>=1.0, transformers>=5.0,
#   sentence-transformers>=5.0
# ============================================================

!pip install -q langchain langchain-core langchain-classic langchain-community langchain-huggingface langchain-text-splitters
!pip install -q sentence-transformers faiss-cpu flashrank transformers torch rank_bm25 numpy

print("=" * 60)
print("ALL PACKAGES INSTALLED SUCCESSFULLY!")
print("=" * 60)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


ALL PACKAGES INSTALLED SUCCESSFULLY!



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# ============================================================
# IMPORT ALL LIBRARIES
# ============================================================

import os
import warnings
import numpy as np
from pprint import pprint

# ── Compatibility patch for langchain v1.x ──
# langchain_core still references langchain.debug / langchain.verbose
# internally, but langchain v1.x removed these module-level attributes.
# Setting them here prevents AttributeError at runtime.
# See: https://github.com/langchain-ai/langchain/issues/19278
import langchain
for attr in ("debug", "verbose", "llm_cache"):
    if not hasattr(langchain, attr):
        setattr(langchain, attr, False)
warnings.filterwarnings("ignore")

# LangChain Core
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

# Text Splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings & Vector Store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# LLM (transformers v5.x: "text2text-generation" was removed, use "text-generation")
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

print("All imports successful!")
print(f"NumPy version: {np.__version__}")
print(f"LangChain version: {langchain.__version__}")
import transformers
print(f"Transformers version: {transformers.__version__}")
import sentence_transformers
print(f"Sentence-Transformers version: {sentence_transformers.__version__}")

All imports successful!
NumPy version: 2.4.4
LangChain version: 1.3.0
Transformers version: 5.8.1
Sentence-Transformers version: 5.5.0


---
## **PART 3: Document Loading**
---

In a real RAG system, you'd load documents from files (PDF, TXT, HTML, etc).  
For this tutorial, we create a **sample knowledge base** about Artificial Intelligence.

> **Key Concept:** A `Document` in LangChain has two parts:
> - `page_content` → the actual text
> - `metadata` → information about the document (source, page number, etc.)

In [2]:
# ============================================================
# CREATE A SAMPLE KNOWLEDGE BASE
# ============================================================
# This text covers multiple AI topics — perfect for demonstrating
# how RAG retrieves relevant sections for different questions.
# ============================================================

sample_text = """
Artificial Intelligence: A Comprehensive Overview

Section 1: What is Artificial Intelligence?
Artificial Intelligence (AI) is the simulation of human intelligence processes by computer systems. These processes include learning (acquiring information and rules for using it), reasoning (using rules to reach approximate or definite conclusions), and self-correction. AI was founded as an academic discipline in 1956 at the Dartmouth Conference, and has experienced several waves of optimism, followed by disappointment and loss of funding known as "AI winters". The field has seen remarkable progress since 2012 due to advances in computing power and availability of large datasets.

Section 2: Machine Learning Fundamentals
Machine Learning (ML) is a subset of AI that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning. First, Supervised Learning where the model learns from labeled training data to make predictions. Examples include classification (spam detection, image recognition) and regression (price prediction, weather forecasting). Second, Unsupervised Learning where the model finds hidden patterns in unlabeled data. Common techniques include clustering (customer segmentation) and dimensionality reduction (PCA). Third, Reinforcement Learning where an agent learns by interacting with an environment, receiving rewards for good actions and penalties for bad ones. Applications include game playing (AlphaGo) and robotics.

Section 3: Deep Learning and Neural Networks
Deep Learning is a subset of machine learning that uses artificial neural networks with multiple layers to model complex patterns. A neural network consists of layers of interconnected nodes (neurons) that process information. Key architectures include Convolutional Neural Networks (CNNs) which are designed for processing grid-like data such as images, Recurrent Neural Networks (RNNs) which handle sequential data like text and time series, and Transformers which use self-attention mechanisms and have revolutionized NLP since 2017. The Transformer architecture, introduced in the paper "Attention is All You Need", eliminated the need for recurrence and convolution, enabling much more efficient parallel processing.

Section 4: Natural Language Processing
Natural Language Processing (NLP) is a branch of AI focused on the interaction between computers and human language. Key NLP tasks include sentiment analysis (determining if text is positive or negative), named entity recognition (identifying people, places, organizations), machine translation (converting text between languages), text summarization (creating concise summaries of long documents), and question answering (finding answers to questions from a given context). Modern NLP heavily relies on large language models like BERT, GPT, T5, and LLaMA which are pre-trained on massive text corpora using self-supervised learning.

Section 5: Computer Vision
Computer Vision enables machines to interpret and make decisions based on visual data. Major applications include facial recognition for security and authentication, autonomous vehicles for detecting roads, obstacles, and traffic signs, medical image analysis for detecting tumors and diseases from X-rays and MRIs, and quality inspection in manufacturing for identifying defects. Deep learning, particularly CNNs like ResNet and EfficientNet, has dramatically improved computer vision accuracy since AlexNet won the ImageNet competition in 2012.

Section 6: Retrieval-Augmented Generation (RAG)
Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by retrieving relevant information from external knowledge bases before generating responses. RAG addresses key limitations of LLMs such as knowledge cutoff dates, hallucination, and lack of domain-specific knowledge. The RAG process involves three main steps: first, indexing where documents are split into chunks, converted to embeddings, and stored in a vector database; second, retrieval where a user query is embedded and similar document chunks are found using similarity search; and third, generation where the retrieved context is combined with the query and fed to an LLM to produce an accurate answer.

Section 7: Vector Databases and Embeddings
Vector databases store data as high-dimensional vectors, enabling efficient similarity search. Text embeddings transform words and sentences into dense numerical vectors that capture semantic meaning. When two texts discuss similar concepts, their embedding vectors will be close together in the vector space, even if they use completely different words. Popular embedding models include BGE (BAAI General Embedding), Sentence-BERT, and E5. Vector databases like FAISS (Facebook AI Similarity Search), Pinecone, Chroma, and Weaviate power modern search and RAG applications. FAISS is an open-source library that enables efficient similarity search in high-dimensional spaces.

Section 8: Transfer Learning and Fine-Tuning
Transfer learning is a technique where a model trained on one task is repurposed for a different but related task. Instead of training from scratch, you start with a pre-trained model and adapt it to your specific use case. Fine-tuning involves taking a pre-trained model and continuing training on a smaller, task-specific dataset. This approach dramatically reduces the data and compute needed compared to training from scratch. Popular approaches include full fine-tuning where all model parameters are updated, LoRA (Low-Rank Adaptation) which adds small trainable matrices, and prompt tuning which optimizes only the input prompts.

Section 9: AI Ethics and Responsible AI
AI ethics involves addressing the moral and social implications of AI systems. Key concerns include bias in training data leading to discriminatory outcomes against certain demographic groups, lack of transparency in AI decision-making often called the black box problem, privacy concerns from extensive data collection and surveillance, potential job displacement as AI automates more tasks, and environmental impact from the massive computational resources needed for training large models. Responsible AI development requires diverse development teams, thorough bias testing, interpretability tools like LIME and SHAP, and regulatory compliance such as the EU AI Act.

Section 10: AI in Healthcare
AI is transforming healthcare through numerous applications. Machine learning models can detect diseases from medical images like X-rays, CT scans, and MRIs with accuracy comparable to experienced radiologists. Natural language processing helps extract insights from electronic health records, clinical notes, and medical literature. Drug discovery has been accelerated by AI models that can predict molecular properties and identify potential drug candidates. Personalized medicine uses patient data to tailor treatments to individual genetic profiles. AI-powered chatbots provide initial triage and health information to patients, reducing the burden on healthcare systems.
"""

document = Document(
    page_content=sample_text,
    metadata={"source": "AI_Textbook", "chapter": "Overview", "author": "Teaching Guide"}
)

print("=" * 60)
print("DOCUMENT LOADED SUCCESSFULLY")
print("=" * 60)
print(f"Document type       : {type(document).__name__}")
print(f"Content length      : {len(document.page_content)} characters")
print(f"Number of words     : {len(document.page_content.split())}")
print(f"Number of sections  : 10")
print(f"Metadata            : {document.metadata}")
print("=" * 60)
print("\nFirst 300 characters of the document:")
print("-" * 60)
print(document.page_content[:300] + "...")

DOCUMENT LOADED SUCCESSFULLY
Document type       : Document
Content length      : 7111 characters
Number of words     : 983
Number of sections  : 10
Metadata            : {'source': 'AI_Textbook', 'chapter': 'Overview', 'author': 'Teaching Guide'}

First 300 characters of the document:
------------------------------------------------------------

Artificial Intelligence: A Comprehensive Overview

Section 1: What is Artificial Intelligence?
Artificial Intelligence (AI) is the simulation of human intelligence processes by computer systems. These processes include learning (acquiring information and rules for using it), reasoning (using rules ...


---
## **PART 4: Text Chunking (Splitting)**
---

### Why chunk documents?

LLMs have a **limited context window** (e.g. 4K, 8K, 128K tokens). We can't feed the entire document.  
Instead, we split it into smaller **chunks** and only retrieve the relevant ones.

### `RecursiveCharacterTextSplitter` — How it works:

```
Full Document
│
├─► Try splitting by "\n\n" (paragraphs)     ← tries this FIRST
├─► Try splitting by "\n" (newlines)
├─► Try splitting by ". " (sentences)
├─► Try splitting by " " (words)
└─► Split by character                         ← last resort
```

**Key Parameters:**
- `chunk_size` → Maximum characters per chunk
- `chunk_overlap` → Characters shared between adjacent chunks (provides context continuity)

In [3]:
# ============================================================
# SPLIT DOCUMENT INTO CHUNKS
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
    add_start_index=True
)

chunks = text_splitter.split_documents([document])

print("=" * 60)
print(f"CHUNKING RESULTS")
print("=" * 60)
print(f"Original document length : {len(document.page_content)} characters")
print(f"Chunk size               : 500 characters")
print(f"Chunk overlap            : 50 characters")
print(f"Number of chunks created : {len(chunks)}")
print("=" * 60)

for i, chunk in enumerate(chunks):
    print(f"\n{'━' * 60}")
    print(f"  CHUNK {i+1}/{len(chunks)}")
    print(f"  Length: {len(chunk.page_content)} chars | Start Index: {chunk.metadata.get('start_index', 'N/A')}")
    print(f"{'━' * 60}")
    print(chunk.page_content)

print(f"\n{'=' * 60}")
print("All chunks displayed above. Each chunk is a self-contained piece of text.")
print(f"{'=' * 60}")

CHUNKING RESULTS
Original document length : 7111 characters
Chunk size               : 500 characters
Chunk overlap            : 50 characters
Number of chunks created : 31

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CHUNK 1/31
  Length: 49 chars | Start Index: 1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Artificial Intelligence: A Comprehensive Overview

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CHUNK 2/31
  Length: 43 chars | Start Index: 52
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Section 1: What is Artificial Intelligence?

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CHUNK 3/31
  Length: 465 chars | Start Index: 96
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Artificial Intelligence (AI) is the simulation of human intelligence processes by computer systems. These processes include learning (acquiring information and rules for using it), reasoning (using rules to reach a

In [4]:
# ============================================================
# VISUALIZE CHUNK OVERLAP
# ============================================================
# Chunk overlap ensures context continuity between chunks.
# The END of chunk N overlaps with the START of chunk N+1.
# ============================================================

print("CHUNK OVERLAP DEMONSTRATION")
print("=" * 60)

if len(chunks) >= 2:
    chunk1_text = chunks[0].page_content
    chunk2_text = chunks[1].page_content

    print(f"\nChunk 1 (last 100 chars):")
    print(f"  ...{chunk1_text[-100:]}")

    print(f"\nChunk 2 (first 100 chars):")
    print(f"  {chunk2_text[:100]}...")

    overlap = ""
    for i in range(min(len(chunk1_text), len(chunk2_text)), 0, -1):
        if chunk1_text.endswith(chunk2_text[:i]):
            overlap = chunk2_text[:i]
            break

    if overlap:
        print(f"\nOverlapping text ({len(overlap)} chars):")
        print(f"  >>> '{overlap}'")
    else:
        print("\nNo exact overlap found (splitter may have adjusted boundaries)")

print("\nWhy overlap matters:")
print("  - Prevents cutting sentences in half")
print("  - Maintains context across chunk boundaries")
print("  - Improves retrieval accuracy")

CHUNK OVERLAP DEMONSTRATION

Chunk 1 (last 100 chars):
  ...Artificial Intelligence: A Comprehensive Overview

Chunk 2 (first 100 chars):
  Section 1: What is Artificial Intelligence?...

No exact overlap found (splitter may have adjusted boundaries)

Why overlap matters:
  - Prevents cutting sentences in half
  - Maintains context across chunk boundaries
  - Improves retrieval accuracy


---
## **PART 5: Understanding Embeddings**
---

### What is an Embedding?

An embedding converts text into a **fixed-size numerical vector** that captures its **semantic meaning**.

```
Text: "Deep learning uses neural networks"
                    │
                    ▼  (Embedding Model)
                    │
Vector: [0.23, -0.45, 0.78, 0.12, ..., -0.33]   ← 384 dimensions
```

### Why Embeddings?

| Text Comparison | Without Embeddings | With Embeddings |
|---|---|---|
| "dog" vs "puppy" | Completely different strings | Very similar vectors (close in space) |
| "bank" (river) vs "bank" (finance) | Same string | Different vectors (far apart in context) |

### Model We Use: `BAAI/bge-small-en-v1.5`

- **BAAI** = Beijing Academy of Artificial Intelligence
- **BGE** = BAAI General Embedding
- **small** = 33M parameters (fast on CPU)
- **384 dimensions** per embedding
- **Open-source** and free to use

In [8]:
# ============================================================
# INITIALIZE THE EMBEDDING MODEL
# ============================================================
# We use BGE-small from BAAI — a top-performing open-source
# embedding model that runs efficiently on CPU.
# First run will download the model (~90MB).
# ============================================================

model_name = "BAAI/bge-small-en-v1.5"

embedding_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("=" * 60)
print("EMBEDDING MODEL LOADED")
print("=" * 60)
print(f"Model name        : {model_name}")
print(f"Model type        : {type(embedding_model).__name__}")
print(f"Device            : CPU")
print(f"Normalization     : True (vectors have unit length)")

test_embedding = embedding_model.embed_query("test")
print(f"Embedding dimension: {len(test_embedding)}")
print("=" * 60)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

EMBEDDING MODEL LOADED
Model name        : BAAI/bge-small-en-v1.5
Model type        : HuggingFaceEmbeddings
Device            : CPU
Normalization     : True (vectors have unit length)
Embedding dimension: 384


In [9]:
# ============================================================
# CHUNK → EMBEDDING: See the transformation in action!
# ============================================================
# Let's take a single chunk and see what its embedding looks like.
# ============================================================

sample_chunk = chunks[0]

print("STEP 1: The Raw Text Chunk")
print("=" * 60)
print(sample_chunk.page_content)
print(f"\nLength: {len(sample_chunk.page_content)} characters")
print(f"Words : {len(sample_chunk.page_content.split())} words")

print("\n\nSTEP 2: The Embedding Vector")
print("=" * 60)

chunk_embedding = embedding_model.embed_query(sample_chunk.page_content)

print(f"Type       : {type(chunk_embedding)}")
print(f"Dimensions : {len(chunk_embedding)}")
print(f"\nFirst 20 values of the embedding vector:")
print(f"  {chunk_embedding[:20]}")
print(f"\nLast 5 values:")
print(f"  {chunk_embedding[-5:]}")
print(f"\nMin value  : {min(chunk_embedding):.6f}")
print(f"Max value  : {max(chunk_embedding):.6f}")
print(f"Mean value : {np.mean(chunk_embedding):.6f}")
print(f"Norm (L2)  : {np.linalg.norm(chunk_embedding):.6f} (should be ~1.0 since normalized)")

print("\n" + "=" * 60)
print("SUMMARY: Text Chunk → 384-dimensional numerical vector!")
print("This vector captures the MEANING of the text.")
print("=" * 60)

STEP 1: The Raw Text Chunk
Artificial Intelligence: A Comprehensive Overview

Length: 49 characters
Words : 5 words


STEP 2: The Embedding Vector
Type       : <class 'list'>
Dimensions : 384

First 20 values of the embedding vector:
  [-0.0139461113139987, -0.000535442610271275, 0.029778841882944107, -0.04741952568292618, 0.004391396883875132, 0.03951618820428848, 0.017003940418362617, 0.04708224534988403, 0.05469680204987526, -0.006212145555764437, 0.02419636957347393, -0.025834666565060616, 0.03245744854211807, 0.030115650966763496, 0.057646725326776505, 0.016128012910485268, 0.013946451246738434, 0.03655778616666794, 0.017473341897130013, -0.040984928607940674]

Last 5 values:
  [0.029739897698163986, -0.041962821036577225, 0.03923029825091362, 0.01704591140151024, -0.04142636060714722]

Min value  : -0.287608
Max value  : 0.273584
Mean value : 0.000373
Norm (L2)  : 1.000000 (should be ~1.0 since normalized)

SUMMARY: Text Chunk → 384-dimensional numerical vector!
This vector captu

In [10]:
# ============================================================
# EMBEDDING SIMILARITY: How similar are different texts?
# ============================================================
# Cosine similarity ranges from -1 (opposite) to 1 (identical).
# Semantically similar texts will have HIGH cosine similarity.
# ============================================================

def cosine_similarity(vec1, vec2):
    vec1, vec2 = np.array(vec1), np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

texts = [
    "Deep learning uses neural networks with multiple layers",
    "Neural networks are a key part of deep learning systems",
    "The weather forecast predicts rain tomorrow afternoon",
    "Machine learning is a subset of artificial intelligence",
]

print("EMBEDDING SIMILARITY COMPARISON")
print("=" * 60)

embeddings = [embedding_model.embed_query(t) for t in texts]

for i, text in enumerate(texts):
    print(f"  Text {i+1}: \"{text}\"")
print()

print("Cosine Similarity Matrix:")
print("-" * 60)
header = "        " + "  ".join([f"Text {i+1}" for i in range(len(texts))])
print(header)

for i in range(len(texts)):
    row = f"Text {i+1}  "
    for j in range(len(texts)):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        row += f"  {sim:.3f}"
    print(row)

print()
sim_12 = cosine_similarity(embeddings[0], embeddings[1])
sim_13 = cosine_similarity(embeddings[0], embeddings[2])
sim_14 = cosine_similarity(embeddings[0], embeddings[3])

print("KEY OBSERVATIONS:")
print(f"  Text 1 vs Text 2 (both about deep learning) : {sim_12:.3f} ← HIGH similarity")
print(f"  Text 1 vs Text 3 (DL vs weather)            : {sim_13:.3f} ← LOW similarity")
print(f"  Text 1 vs Text 4 (DL vs ML/AI)              : {sim_14:.3f} ← MODERATE similarity")
print("\nThis is how embeddings capture semantic meaning!")

EMBEDDING SIMILARITY COMPARISON
  Text 1: "Deep learning uses neural networks with multiple layers"
  Text 2: "Neural networks are a key part of deep learning systems"
  Text 3: "The weather forecast predicts rain tomorrow afternoon"
  Text 4: "Machine learning is a subset of artificial intelligence"

Cosine Similarity Matrix:
------------------------------------------------------------
        Text 1  Text 2  Text 3  Text 4
Text 1    1.000  0.899  0.466  0.715
Text 2    0.899  1.000  0.474  0.812
Text 3    0.466  0.474  1.000  0.494
Text 4    0.715  0.812  0.494  1.000

KEY OBSERVATIONS:
  Text 1 vs Text 2 (both about deep learning) : 0.899 ← HIGH similarity
  Text 1 vs Text 3 (DL vs weather)            : 0.466 ← LOW similarity
  Text 1 vs Text 4 (DL vs ML/AI)              : 0.715 ← MODERATE similarity

This is how embeddings capture semantic meaning!


---
## **PART 6: Vector Store — Where Embeddings Live**
---

### What is a Vector Store?

A vector store is a **database optimized for storing and searching embedding vectors**.  
When you add documents, it:
1. Converts each chunk to an embedding vector
2. Stores the vector alongside the original text
3. Builds an **index** for fast similarity search

### FAISS (Facebook AI Similarity Search)

We use **FAISS** — an open-source library by Meta for efficient similarity search.

| Feature | Value |
|---|---|
| Storage | **In-memory** (RAM) |
| Index type | Flat L2 (exact search) |
| Speed | Very fast for < 1M vectors |
| Cost | Free, open-source |

> In production, you might use Pinecone, Chroma, Weaviate, or Qdrant for persistent storage.

In [11]:
# ============================================================
# CREATE THE VECTOR STORE (FAISS)
# ============================================================
# This step embeds ALL chunks and stores them in FAISS.
# FAISS stores everything IN-MEMORY (RAM).
# ============================================================

import time

print("Creating vector store from chunks...")
print(f"Number of chunks to embed: {len(chunks)}")
print()

start_time = time.time()
vectorstore = FAISS.from_documents(chunks, embedding_model)
elapsed = time.time() - start_time

print("=" * 60)
print("VECTOR STORE CREATED SUCCESSFULLY!")
print("=" * 60)
print(f"Number of vectors stored  : {vectorstore.index.ntotal}")
print(f"Vector dimension          : {vectorstore.index.d}")
print(f"Index type                : {type(vectorstore.index).__name__}")
print(f"Time to create            : {elapsed:.2f} seconds")
print(f"Storage location          : IN-MEMORY (RAM)")
print(f"Vectorstore type          : {type(vectorstore).__name__}")
print("=" * 60)

print("\nWhere is the data stored?")
print("-" * 60)
print(f"  vectorstore object ID   : {id(vectorstore)}")
print(f"  FAISS index object      : {vectorstore.index}")
print(f"  Document store          : {type(vectorstore.docstore).__name__}")
print(f"  Number of docs in store : {len(vectorstore.docstore._dict)}")
print("\n  The embeddings live in RAM as a numpy array inside the FAISS index.")
print("  The original text is stored in the docstore (a Python dictionary).")

Creating vector store from chunks...
Number of chunks to embed: 31

VECTOR STORE CREATED SUCCESSFULLY!
Number of vectors stored  : 31
Vector dimension          : 384
Index type                : IndexFlatL2
Time to create            : 1.82 seconds
Storage location          : IN-MEMORY (RAM)
Vectorstore type          : FAISS

Where is the data stored?
------------------------------------------------------------
  vectorstore object ID   : 138448183756560
  FAISS index object      : <faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x7deb524b0090> >
  Document store          : InMemoryDocstore
  Number of docs in store : 31

  The embeddings live in RAM as a numpy array inside the FAISS index.
  The original text is stored in the docstore (a Python dictionary).


In [12]:
# ============================================================
# INSPECT WHAT'S INSIDE THE VECTOR STORE
# ============================================================

print("INSIDE THE VECTOR STORE")
print("=" * 60)

doc_ids = list(vectorstore.docstore._dict.keys())
print(f"\nStored Document IDs (first 5):")
for i, doc_id in enumerate(doc_ids[:5]):
    doc = vectorstore.docstore._dict[doc_id]
    print(f"  [{i+1}] ID: {doc_id[:20]}...")
    print(f"      Text: \"{doc.page_content[:80]}...\"")
    print(f"      Metadata: {doc.metadata}")
    print()

print("-" * 60)
print(f"Total documents in store: {len(doc_ids)}")
print(f"Total vectors in index  : {vectorstore.index.ntotal}")
print(f"Vector dimension        : {vectorstore.index.d}")

import sys
print(f"\nApproximate memory usage:")
vector_memory = vectorstore.index.ntotal * vectorstore.index.d * 4  # 4 bytes per float32
print(f"  Vectors: {vector_memory / 1024:.1f} KB ({vectorstore.index.ntotal} vectors x {vectorstore.index.d} dims x 4 bytes)")
print(f"  (Text and metadata stored separately in Python dict)")

INSIDE THE VECTOR STORE

Stored Document IDs (first 5):
  [1] ID: ba6fe840-eaa2-454e-a...
      Text: "Artificial Intelligence: A Comprehensive Overview..."
      Metadata: {'source': 'AI_Textbook', 'chapter': 'Overview', 'author': 'Teaching Guide', 'start_index': 1}

  [2] ID: 53e976b6-dc72-47bb-a...
      Text: "Section 1: What is Artificial Intelligence?..."
      Metadata: {'source': 'AI_Textbook', 'chapter': 'Overview', 'author': 'Teaching Guide', 'start_index': 52}

  [3] ID: b462d07c-e0ef-42e3-8...
      Text: "Artificial Intelligence (AI) is the simulation of human intelligence processes b..."
      Metadata: {'source': 'AI_Textbook', 'chapter': 'Overview', 'author': 'Teaching Guide', 'start_index': 96}

  [4] ID: abd3fe08-f713-4afd-a...
      Text: ". The field has seen remarkable progress since 2012 due to advances in computing..."
      Metadata: {'source': 'AI_Textbook', 'chapter': 'Overview', 'author': 'Teaching Guide', 'start_index': 561}

  [5] ID: 5575cd49-aa51-42e8-8..

---
## **PART 7: Retrieval — Finding Relevant Chunks**
---

### How Retrieval Works

```
User Query: "What is deep learning?"
       │
       ▼
Embed the query → [0.4, 0.7, ..., 0.2]     (same embedding model)
       │
       ▼
Compare with ALL stored vectors              (cosine similarity)
       │
       ▼
Return Top-K most similar chunks             (K=4 by default)
```

### LangChain Retriever

A **Retriever** is a LangChain abstraction that:
- Takes a **string query** as input
- Returns a **list of Documents** as output

You can create a retriever from any vector store using `.as_retriever()`

In [13]:
# ============================================================
# RETRIEVAL: Find relevant chunks for a query
# ============================================================

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

query = "What is deep learning and how does it work?"

print("RETRIEVAL DEMONSTRATION")
print("=" * 60)
print(f"Query: \"{query}\"")
print(f"Retrieving top {retriever.search_kwargs['k']} most relevant chunks...")
print("=" * 60)

retrieved_docs = retriever.invoke(query)

print(f"\nNumber of chunks retrieved: {len(retrieved_docs)}")

for i, doc in enumerate(retrieved_docs):
    print(f"\n{'━' * 60}")
    print(f"  RETRIEVED CHUNK {i+1}")
    print(f"  Source: {doc.metadata.get('source', 'N/A')}")
    print(f"  Start Index: {doc.metadata.get('start_index', 'N/A')}")
    print(f"{'━' * 60}")
    print(doc.page_content)

print(f"\n{'=' * 60}")
print("Notice how the retriever found chunks about deep learning")
print("and neural networks — the most relevant to our query!")
print(f"{'=' * 60}")

RETRIEVAL DEMONSTRATION
Query: "What is deep learning and how does it work?"
Retrieving top 4 most relevant chunks...

Number of chunks retrieved: 4

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  RETRIEVED CHUNK 1
  Source: AI_Textbook
  Start Index: 1570
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Deep Learning is a subset of machine learning that uses artificial neural networks with multiple layers to model complex patterns. A neural network consists of layers of interconnected nodes (neurons) that process information

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  RETRIEVED CHUNK 2
  Source: AI_Textbook
  Start Index: 1525
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Section 3: Deep Learning and Neural Networks

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  RETRIEVED CHUNK 3
  Source: AI_Textbook
  Start Index: 3373
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
. Deep learning, particul

In [14]:
# ============================================================
# RETRIEVAL WITH SIMILARITY SCORES
# ============================================================
# similarity_search_with_score() returns both documents AND scores.
# Lower L2 distance = MORE similar (for FAISS with L2 metric).
# ============================================================

query = "What is deep learning and how does it work?"

results_with_scores = vectorstore.similarity_search_with_score(query, k=6)

print("RETRIEVAL WITH SIMILARITY SCORES")
print("=" * 60)
print(f"Query: \"{query}\"")
print(f"Showing top 6 results with their distance scores")
print("(Lower distance = Higher relevance)")
print("=" * 60)

for i, (doc, score) in enumerate(results_with_scores):
    relevance = "HIGH" if score < 0.8 else ("MEDIUM" if score < 1.2 else "LOW")
    print(f"\n--- Rank {i+1} | L2 Distance: {score:.4f} | Relevance: {relevance} ---")
    print(f"Text: {doc.page_content[:200]}...")
    print(f"Metadata: {doc.metadata}")

print(f"\n{'=' * 60}")
print("Observe how scores increase (less relevant) as we go down the list.")
print("The retriever returns chunks ordered by relevance!")
print(f"{'=' * 60}")

RETRIEVAL WITH SIMILARITY SCORES
Query: "What is deep learning and how does it work?"
Showing top 6 results with their distance scores
(Lower distance = Higher relevance)

--- Rank 1 | L2 Distance: 0.3291 | Relevance: HIGH ---
Text: Deep Learning is a subset of machine learning that uses artificial neural networks with multiple layers to model complex patterns. A neural network consists of layers of interconnected nodes (neurons)...
Metadata: {'source': 'AI_Textbook', 'chapter': 'Overview', 'author': 'Teaching Guide', 'start_index': 1570}

--- Rank 2 | L2 Distance: 0.5344 | Relevance: HIGH ---
Text: Section 3: Deep Learning and Neural Networks...
Metadata: {'source': 'AI_Textbook', 'chapter': 'Overview', 'author': 'Teaching Guide', 'start_index': 1525}

--- Rank 3 | L2 Distance: 0.5415 | Relevance: HIGH ---
Text: . Deep learning, particularly CNNs like ResNet and EfficientNet, has dramatically improved computer vision accuracy since AlexNet won the ImageNet competition in 2012....
Meta

In [15]:
# ============================================================
# TRY DIFFERENT QUERIES — See what gets retrieved!
# ============================================================

queries = [
    "What are the ethical concerns in AI?",
    "How is AI used in healthcare?",
    "What is transfer learning and fine-tuning?",
    "Explain RAG and vector databases",
]

print("MULTI-QUERY RETRIEVAL TEST")
print("=" * 60)

for query in queries:
    print(f"\n{'━' * 60}")
    print(f"QUERY: \"{query}\"")
    print(f"{'━' * 60}")

    results = vectorstore.similarity_search_with_score(query, k=2)

    for i, (doc, score) in enumerate(results):
        print(f"  [{i+1}] Score: {score:.4f}")
        print(f"      Chunk: \"{doc.page_content[:120]}...\"")
        print()

print("=" * 60)
print("Each query retrieves DIFFERENT chunks — exactly the relevant ones!")
print("=" * 60)

MULTI-QUERY RETRIEVAL TEST

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
QUERY: "What are the ethical concerns in AI?"
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [1] Score: 0.2181
      Chunk: "AI ethics involves addressing the moral and social implications of AI systems. Key concerns include bias in training dat..."

  [2] Score: 0.3149
      Chunk: "Section 9: AI Ethics and Responsible AI..."


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
QUERY: "How is AI used in healthcare?"
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [1] Score: 0.3177
      Chunk: "Section 10: AI in Healthcare..."

  [2] Score: 0.3666
      Chunk: "AI is transforming healthcare through numerous applications. Machine learning models can detect diseases from medical im..."


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
QUERY: "What is transfer learning and fine-tuning?"
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


---
## **PART 8: Understanding Chains & RunnablePassthrough**
---

### What is a Chain?

A **Chain** in LangChain is a **sequence of processing steps** connected together, where the output of one step feeds into the next.

```
Input ──► Step 1 ──► Step 2 ──► Step 3 ──► Output
          (retrieve)  (format)   (LLM)     (answer)
```

In LangChain, chains are built using the **LCEL (LangChain Expression Language)** with the pipe operator (`|`).

### What is `RunnablePassthrough`?

`RunnablePassthrough` is a special component that **passes its input through unchanged**. It's used when you need to forward the original input alongside transformed data.

```python
# Without RunnablePassthrough:
chain = retriever | format_docs | prompt | llm    # ← lost the original query!

# With RunnablePassthrough:
chain = {
    "context": retriever | format_docs,      # ← transforms input to context
    "question": RunnablePassthrough()         # ← passes input as-is (the query)
} | prompt | llm
```

### What is `RunnableParallel`?

`RunnableParallel` runs multiple operations **simultaneously** and collects their outputs into a dictionary.  
When you write `{"key1": runnable1, "key2": runnable2}`, LangChain automatically wraps it in `RunnableParallel`.

### What is `RunnableLambda`?

`RunnableLambda` wraps a **plain Python function** into a LangChain runnable so it can be used in chains.

In [16]:
# ============================================================
# DEMO: Understanding RunnablePassthrough & Chain Components
# ============================================================

print("DEMO 1: RunnablePassthrough — passes input unchanged")
print("=" * 60)

passthrough = RunnablePassthrough()
result = passthrough.invoke("Hello, I am the original input!")
print(f"Input:  'Hello, I am the original input!'")
print(f"Output: '{result}'")
print("→ The input passes through unchanged!\n")

print("DEMO 2: RunnableLambda — wraps a function")
print("=" * 60)

uppercase_fn = RunnableLambda(lambda x: x.upper())
result = uppercase_fn.invoke("hello world")
print(f"Input:  'hello world'")
print(f"Output: '{result}'")
print("→ The function transforms the input!\n")

print("DEMO 3: Chaining with | (pipe operator)")
print("=" * 60)

chain = RunnableLambda(lambda x: x.upper()) | RunnableLambda(lambda x: f"*** {x} ***")
result = chain.invoke("hello")
print(f"Input:  'hello'")
print(f"Step 1: UPPER → 'HELLO'")
print(f"Step 2: DECORATE → '{result}'")
print("→ Each step feeds into the next!\n")

print("DEMO 4: RunnableParallel — runs steps in parallel")
print("=" * 60)

parallel = RunnableParallel(
    original=RunnablePassthrough(),
    uppercased=RunnableLambda(lambda x: x.upper()),
    length=RunnableLambda(lambda x: len(x))
)
result = parallel.invoke("hello world")
print(f"Input: 'hello world'")
print(f"Output (dict):")
for key, value in result.items():
    print(f"  '{key}' → {value}")
print("→ Same input processed in parallel, different outputs collected!\n")

print("DEMO 5: How it works in RAG")
print("=" * 60)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_input_chain = RunnableParallel(
    context=retriever | RunnableLambda(format_docs),
    question=RunnablePassthrough()
)

result = rag_input_chain.invoke("What is deep learning?")
print(f"Input query: 'What is deep learning?'")
print(f"\nOutput keys: {list(result.keys())}")
print(f"\n'question' value: '{result['question']}'")
print(f"\n'context' value (first 300 chars):")
print(f"  {result['context'][:300]}...")
print("\n→ The query is passed through AND used to retrieve context!")

DEMO 1: RunnablePassthrough — passes input unchanged
Input:  'Hello, I am the original input!'
Output: 'Hello, I am the original input!'
→ The input passes through unchanged!

DEMO 2: RunnableLambda — wraps a function
Input:  'hello world'
Output: 'HELLO WORLD'
→ The function transforms the input!

DEMO 3: Chaining with | (pipe operator)
Input:  'hello'
Step 1: UPPER → 'HELLO'
Step 2: DECORATE → '*** HELLO ***'
→ Each step feeds into the next!

DEMO 4: RunnableParallel — runs steps in parallel
Input: 'hello world'
Output (dict):
  'original' → hello world
  'uppercased' → HELLO WORLD
  'length' → 11
→ Same input processed in parallel, different outputs collected!

DEMO 5: How it works in RAG
Input query: 'What is deep learning?'

Output keys: ['context', 'question']

'question' value: 'What is deep learning?'

'context' value (first 300 chars):
  Deep Learning is a subset of machine learning that uses artificial neural networks with multiple layers to model complex patterns. A neural n

---
## **PART 9: Building the Complete RAG Pipeline**
---

Now we connect everything into a working RAG system:

```
Query ──► Retriever ──► Format Docs ──► Prompt Template ──► LLM ──► Answer
  │                          │                │               │
  │         RunnablePassthrough ──────────► question         │
  │                          │                │               │
  └─────────── "What is DL?" │   "Context: {chunks}         │
                             │    Question: What is DL?      │
                             │    Answer: ..."               │
                             └────────────────────────────────┘
```

### LLM: `google/flan-t5-small`

We use **Flan-T5-Small** — a small but capable open-source text-to-text model by Google.  
- Size: ~300MB  
- Runs on CPU  
- No API key needed

In [17]:
# ============================================================
# SETUP THE LLM (Language Model)
# ============================================================
# We use google/flan-t5-small — a small open-source model.
# First run will download the model (~300MB).
# ============================================================

model_id = "google/flan-t5-small"

print(f"Loading LLM: {model_id}")
print("(This may take a minute on first run as the model downloads...)")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.1
)

llm = HuggingFacePipeline(pipeline=pipe)

print("\n" + "=" * 60)
print("LLM LOADED SUCCESSFULLY!")
print("=" * 60)
print(f"Model           : {model_id}")
print(f"Type            : Text-to-Text Generation (Seq2Seq)")
print(f"Parameters      : ~80M")
print(f"Max new tokens  : 256")
print(f"Device          : CPU")
print(f"API key needed  : No (fully open-source)")
print("=" * 60)

print("\nQuick test — asking the LLM directly (without RAG):")
test_answer = llm.invoke("What is deep learning?")
print(f"  Q: What is deep learning?")
print(f"  A: {test_answer}")
print("\n(Notice: Without RAG context, the answer is very brief!)")

Loading LLM: google/flan-t5-small
(This may take a minute on first run as the model downloads...)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3


LLM LOADED SUCCESSFULLY!
Model           : google/flan-t5-small
Type            : Text-to-Text Generation (Seq2Seq)
Parameters      : ~80M
Max new tokens  : 256
Device          : CPU
API key needed  : No (fully open-source)

Quick test — asking the LLM directly (without RAG):
  Q: What is deep learning?
  A: What is deep learning?

(Notice: Without RAG context, the answer is very brief!)


In [18]:
# ============================================================
# BUILD THE RAG CHAIN
# ============================================================
# This is where everything comes together!
# ============================================================

template = """Answer the question based ONLY on the following context. 
If the context doesn't contain the answer, say "I don't have enough information to answer this."

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate.from_template(template)

print("PROMPT TEMPLATE")
print("=" * 60)
print(prompt.template)
print("=" * 60)

def format_docs(docs):
    """Join document contents with double newlines."""
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("\nRAG CHAIN STRUCTURE:")
print("-" * 60)
print("  Step 1: RunnableParallel")
print("          ├── 'context': retriever → format_docs")
print("          └── 'question': RunnablePassthrough()")
print("  Step 2: PromptTemplate (fills in context + question)")
print("  Step 3: LLM (generates answer)")
print("  Step 4: StrOutputParser (extracts text)")
print("-" * 60)
print("\nRAG chain is ready! Let's use it.")

PROMPT TEMPLATE
Answer the question based ONLY on the following context. 
If the context doesn't contain the answer, say "I don't have enough information to answer this."

Context:
{context}

Question: {question}

Answer:

RAG CHAIN STRUCTURE:
------------------------------------------------------------
  Step 1: RunnableParallel
          ├── 'context': retriever → format_docs
          └── 'question': RunnablePassthrough()
  Step 2: PromptTemplate (fills in context + question)
  Step 3: LLM (generates answer)
  Step 4: StrOutputParser (extracts text)
------------------------------------------------------------

RAG chain is ready! Let's use it.


In [19]:
# ============================================================
# RUN THE RAG PIPELINE — See everything that happens!
# ============================================================
# We'll show BOTH the retrieved chunks AND the final answer.
# ============================================================

query = "What is deep learning and how does it relate to neural networks?"

print("COMPLETE RAG PIPELINE EXECUTION")
print("=" * 70)
print(f"USER QUERY: \"{query}\"")
print("=" * 70)

# Step 1: Retrieve relevant chunks
print("\n[STEP 1] RETRIEVAL — Finding relevant chunks...")
print("-" * 70)
retrieved = retriever.invoke(query)
for i, doc in enumerate(retrieved):
    print(f"  Chunk {i+1}: \"{doc.page_content[:150]}...\"")
print(f"  → Retrieved {len(retrieved)} chunks")

# Step 2: Format context
print("\n[STEP 2] FORMAT — Combining chunks into context...")
print("-" * 70)
context = format_docs(retrieved)
print(f"  Context length: {len(context)} characters")
print(f"  Preview: \"{context[:200]}...\"")

# Step 3: Fill prompt template
print("\n[STEP 3] PROMPT — Filling the template...")
print("-" * 70)
filled_prompt = prompt.format(context=context, question=query)
print(f"  Prompt length: {len(filled_prompt)} characters")
print(f"  (Full prompt sent to the LLM)")

# Step 4: Generate answer
print("\n[STEP 4] GENERATION — LLM produces the answer...")
print("-" * 70)
answer = rag_chain.invoke(query)
print(f"\n{'█' * 70}")
print(f"  FINAL ANSWER: {answer}")
print(f"{'█' * 70}")

print("\n[COMPARISON] Without RAG:")
print("-" * 70)
direct_answer = llm.invoke(query)
print(f"  Direct LLM answer: {direct_answer}")
print("\n→ RAG provides a much more detailed and grounded answer!")

COMPLETE RAG PIPELINE EXECUTION
USER QUERY: "What is deep learning and how does it relate to neural networks?"

[STEP 1] RETRIEVAL — Finding relevant chunks...
----------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Chunk 1: "Deep Learning is a subset of machine learning that uses artificial neural networks with multiple layers to model complex patterns. A neural network co..."
  Chunk 2: "Section 3: Deep Learning and Neural Networks..."
  Chunk 3: ". Deep learning, particularly CNNs like ResNet and EfficientNet, has dramatically improved computer vision accuracy since AlexNet won the ImageNet com..."
  Chunk 4: ". Key architectures include Convolutional Neural Networks (CNNs) which are designed for processing grid-like data such as images, Recurrent Neural Net..."
  → Retrieved 4 chunks

[STEP 2] FORMAT — Combining chunks into context...
----------------------------------------------------------------------
  Context length: 938 characters
  Preview: "Deep Learning is a subset of machine learning that uses artificial neural networks with multiple layers to model complex patterns. A neural network consists of layers of interconnected nodes (neurons)..."

[STEP 3] PROMPT — Filling the template...

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



██████████████████████████████████████████████████████████████████████
  FINAL ANSWER: Answer the question based ONLY on the following context. 
If the context doesn't contain the answer, say "I don't have enough information to answer this."

Context:
Deep Learning is a subset of machine learning that uses artificial neural networks with multiple layers to model complex patterns. A neural network consists of layers of interconnected nodes (neurons) that process information

Section 3: Deep Learning and Neural Networks

. Deep learning, particularly CNNs like ResNet and EfficientNet, has dramatically improved computer vision accuracy since AlexNet won the ImageNet competition in 2012.

. Key architectures include Convolutional Neural Networks (CNNs) which are designed for processing grid-like data such as images, Recurrent Neural Networks (RNNs) which handle sequential data like text and time series, and Transformers which use self-attention mechanisms and have revolutionized NLP since

In [20]:
# ============================================================
# TEST RAG WITH MULTIPLE QUERIES
# ============================================================

queries = [
    "What is retrieval augmented generation?",
    "What are the ethical concerns about AI?",
    "How is AI transforming healthcare?",
    "What is the difference between supervised and unsupervised learning?",
    "What are vector databases used for?",
]

print("RAG PIPELINE — MULTIPLE QUERIES")
print("=" * 70)

for i, query in enumerate(queries):
    print(f"\n{'━' * 70}")
    print(f"  Q{i+1}: {query}")
    print(f"{'━' * 70}")

    # Show retrieved chunks
    retrieved = retriever.invoke(query)
    print(f"  Retrieved chunks:")
    for j, doc in enumerate(retrieved[:2]):
        print(f"    [{j+1}] \"{doc.page_content[:100]}...\"")

    # Get RAG answer
    answer = rag_chain.invoke(query)
    print(f"\n  ANSWER: {answer}")

print(f"\n{'=' * 70}")
print("Each query retrieves different chunks and produces a contextual answer!")
print(f"{'=' * 70}")

RAG PIPELINE — MULTIPLE QUERIES

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Q1: What is retrieval augmented generation?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Retrieved chunks:
    [1] "Section 6: Retrieval-Augmented Generation (RAG)..."
    [2] "Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by retrievin..."


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



  ANSWER: Answer the question based ONLY on the following context. 
If the context doesn't contain the answer, say "I don't have enough information to answer this."

Context:
Section 6: Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by retrieving relevant information from external knowledge bases before generating responses. RAG addresses key limitations of LLMs such as knowledge cutoff dates, hallucination, and lack of domain-specific knowledge

Section 7: Vector Databases and Embeddings

Section 8: Transfer Learning and Fine-Tuning

Question: What is retrieval augmented generation?

Answer:

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Q2: What are the ethical concerns about AI?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Retrieved chunks:
    [1] "AI ethics involves addressing the moral and social implications of AI systems. Key concerns include ..

...


  ANSWER: Answer the question based ONLY on the following context. 
If the context doesn't contain the answer, say "I don't have enough information to answer this."

Context:
AI ethics involves addressing the moral and social implications of AI systems. Key concerns include bias in training data leading to discriminatory outcomes against certain demographic groups, lack of transparency in AI decision-making often called the black box problem, privacy concerns from extensive data collection and surveillance, potential job displacement as AI automates more tasks, and environmental impact from the massive computational resources needed for training large models

Section 9: AI Ethics and Responsible AI

. Responsible AI development requires diverse development teams, thorough bias testing, interpretability tools like LIME and SHAP, and regulatory compliance such as the EU AI Act.

Section 10: AI in Healthcare

Question: What are the ethical concerns about AI?

Answer:

━━━━━━━━━━━━━━━━━━━

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



  ANSWER: Answer the question based ONLY on the following context. 
If the context doesn't contain the answer, say "I don't have enough information to answer this."

Context:
Section 10: AI in Healthcare

AI is transforming healthcare through numerous applications. Machine learning models can detect diseases from medical images like X-rays, CT scans, and MRIs with accuracy comparable to experienced radiologists. Natural language processing helps extract insights from electronic health records, clinical notes, and medical literature. Drug discovery has been accelerated by AI models that can predict molecular properties and identify potential drug candidates

. Personalized medicine uses patient data to tailor treatments to individual genetic profiles. AI-powered chatbots provide initial triage and health information to patients, reducing the burden on healthcare systems.

Section 1: What is Artificial Intelligence?

Question: How is AI transforming healthcare?

Answer:

━━━━━━━━━━━━━━━

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



  ANSWER: Answer the question based ONLY on the following context. 
If the context doesn't contain the answer, say "I don't have enough information to answer this."

Context:
Machine Learning (ML) is a subset of AI that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning. First, Supervised Learning where the model learns from labeled training data to make predictions. Examples include classification (spam detection, image recognition) and regression (price prediction, weather forecasting). Second, Unsupervised Learning where the model finds hidden patterns in unlabeled data

Section 2: Machine Learning Fundamentals

Section 7: Vector Databases and Embeddings

Transfer learning is a technique where a model trained on one task is repurposed for a different but related task. Instead of training from scratch, you start with a pre-trained model and adapt it to your specific use case. Fine-tuning involves t

---
## **PART 10: Introduction to Rerankers**
---

### The Problem with Basic Retrieval

Basic vector similarity search is **fast but imprecise**:
- It uses **bi-encoder** models (embed query & docs separately)
- Cannot capture fine-grained query-document interactions
- May return chunks that are **topically related but don't answer the question**

### What is a Reranker?

A **reranker** is a model that **re-scores retrieved documents** by looking at the query and document **together**.

```
Without Reranking:
  Query ──► Vector Search ──► [Chunk A (0.89), Chunk B (0.85), Chunk C (0.82)] ──► LLM
                                   ↑ might not be the best order!

With Reranking:
  Query ──► Vector Search ──► [Chunk A, B, C, D, E] ──► Reranker ──► [Chunk C, A, B] ──► LLM
                               retrieve MORE (k=20)     re-score      keep TOP (n=3)
                                                        & reorder
```

### Bi-Encoder vs Cross-Encoder

| Feature | Bi-Encoder (Retrieval) | Cross-Encoder (Reranking) |
|---|---|---|
| How it works | Embeds query & doc separately | Processes query+doc together |
| Speed | Very fast (pre-computed) | Slower (computes per pair) |
| Accuracy | Good | Better |
| Use case | First-stage retrieval (1000→20) | Second-stage reranking (20→3) |

### Rerankers We'll Cover:

1. **FlashRank** — Ultra-fast, lightweight reranker
2. **Cross-Encoder** — SoTA accuracy using sentence-transformers
3. **BM25 + Hybrid Search** — Keyword-based complement to semantic search

---
### **Reranker 1: FlashRank**
---

[FlashRank](https://github.com/PrithivirajDamodaran/FlashRank) is an ultra-lite and super-fast reranker.

- **Tiny model size** (~4MB)
- **No GPU needed** — runs great on CPU
- Based on cross-encoder architecture
- Integrated with LangChain via `ContextualCompressionRetriever`

In [21]:
# ============================================================
# RERANKER 1: FlashRank
# ============================================================

# LangChain v1.x: retrievers moved to langchain_classic
# See: https://docs.langchain.com/oss/python/migrate/langchain-v1
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

compressor = FlashrankRerank(top_n=3)

flashrank_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

query = "What is deep learning and how does it work?"

print("FLASHRANK RERANKER DEMONSTRATION")
print("=" * 70)
print(f"Query: \"{query}\"")
print()

# Step 1: Base retrieval (top 10)
print("[STEP 1] Base retriever fetches top 10 chunks:")
print("-" * 70)
base_results = base_retriever.invoke(query)
for i, doc in enumerate(base_results):
    print(f"  [{i+1:2d}] \"{doc.page_content[:100]}...\"")

# Step 2: Reranking (down to top 3)
print(f"\n[STEP 2] FlashRank reranks and selects top 3:")
print("-" * 70)
reranked_results = flashrank_retriever.invoke(query)
for i, doc in enumerate(reranked_results):
    score = doc.metadata.get("relevance_score", "N/A")
    print(f"  [{i+1}] Relevance Score: {score}")
    print(f"      \"{doc.page_content[:150]}...\"")
    print()

print("=" * 70)
print("FlashRank re-scored all 10 chunks and returned the best 3!")
print("Notice the relevance_score in metadata — higher = more relevant.")
print("=" * 70)

INFO:flashrank.Ranker:Downloading ms-marco-MultiBERT-L-12...
ms-marco-MultiBERT-L-12.zip: 100%|████████████████████████████████████| 98.7M/98.7M [00:27<00:00, 3.81MiB/s]


FLASHRANK RERANKER DEMONSTRATION
Query: "What is deep learning and how does it work?"

[STEP 1] Base retriever fetches top 10 chunks:
----------------------------------------------------------------------
  [ 1] "Deep Learning is a subset of machine learning that uses artificial neural networks with multiple lay..."
  [ 2] "Section 3: Deep Learning and Neural Networks..."
  [ 3] ". Deep learning, particularly CNNs like ResNet and EfficientNet, has dramatically improved computer ..."
  [ 4] ". Key architectures include Convolutional Neural Networks (CNNs) which are designed for processing g..."
  [ 5] "Machine Learning (ML) is a subset of AI that enables systems to learn and improve from experience wi..."
  [ 6] "AI is transforming healthcare through numerous applications. Machine learning models can detect dise..."
  [ 7] "Natural Language Processing (NLP) is a branch of AI focused on the interaction between computers and..."
  [ 8] "Transfer learning is a technique where a model train

In [22]:
# ============================================================
# COMPARISON: With vs Without FlashRank Reranking
# ============================================================

query = "How does RAG work with vector databases?"

print("SIDE-BY-SIDE COMPARISON")
print("=" * 70)
print(f"Query: \"{query}\"")

# Without reranking (basic top 3)
basic_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
basic_results = basic_retriever.invoke(query)

print(f"\n--- WITHOUT Reranking (top 3 by vector similarity) ---")
for i, doc in enumerate(basic_results):
    print(f"  [{i+1}] \"{doc.page_content[:120]}...\"")

# With FlashRank reranking
reranked_results = flashrank_retriever.invoke(query)

print(f"\n--- WITH FlashRank Reranking (top 3 after re-scoring) ---")
for i, doc in enumerate(reranked_results):
    score = doc.metadata.get("relevance_score", "N/A")
    print(f"  [{i+1}] Score: {score}")
    print(f"      \"{doc.page_content[:120]}...\"")

print(f"\n{'=' * 70}")
print("The reranker often reorders chunks, bringing more relevant ones to the top!")
print("It retrieves 10 candidates first, then picks the best 3.")
print(f"{'=' * 70}")

SIDE-BY-SIDE COMPARISON
Query: "How does RAG work with vector databases?"

--- WITHOUT Reranking (top 3 by vector similarity) ---
  [1] ". The RAG process involves three main steps: first, indexing where documents are split into chunks, converted to embeddi..."
  [2] "Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by retrieving relevant informati..."
  [3] "Section 6: Retrieval-Augmented Generation (RAG)..."

--- WITH FlashRank Reranking (top 3 after re-scoring) ---
  [1] Score: 0.9967000484466553
      ". Vector databases like FAISS (Facebook AI Similarity Search), Pinecone, Chroma, and Weaviate power modern search and RA..."
  [2] Score: 0.995409369468689
      "Section 7: Vector Databases and Embeddings..."
  [3] Score: 0.988429844379425
      "Vector databases store data as high-dimensional vectors, enabling efficient similarity search. Text embeddings transform..."

The reranker often reorders chunks, bringing more relevant ones to the top!

---
### **Reranker 2: Cross-Encoder**
---

A **Cross-Encoder** processes the query and document **together** as a single input,  
producing a relevance score. This is more accurate but slower than bi-encoders.

```
Bi-Encoder (used in retrieval):
  Query ──► Encoder ──► query_vec ─┐
                                    ├──► cosine_similarity → score
  Doc   ──► Encoder ──► doc_vec  ──┘

Cross-Encoder (used in reranking):
  [Query + Doc] ──► Encoder ──► relevance_score
  (processed together — captures interactions!)
```

**Model:** `cross-encoder/ms-marco-MiniLM-L-6-v2`
- Trained on MS MARCO passage ranking dataset
- Small and efficient (~80MB)
- Great accuracy for reranking

In [23]:
# ============================================================
# RERANKER 2: Cross-Encoder
# ============================================================

# HuggingFaceCrossEncoder lives in langchain-community
# CrossEncoderReranker lives in langchain-classic (v1.x)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

print("Loading Cross-Encoder model...")
cross_encoder_model = HuggingFaceCrossEncoder(
    model_name="cross-encoder/ms-marco-MiniLM-L-6-v2"
)
print("Cross-Encoder model loaded!\n")

cross_encoder_compressor = CrossEncoderReranker(
    model=cross_encoder_model,
    top_n=3
)

cross_encoder_retriever = ContextualCompressionRetriever(
    base_compressor=cross_encoder_compressor,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": 10})
)

query = "What is deep learning and how does it work?"

print("CROSS-ENCODER RERANKER DEMONSTRATION")
print("=" * 70)
print(f"Query: \"{query}\"")

# Base retrieval
print(f"\n[STEP 1] Base retriever fetches top 10 chunks")
base_results = vectorstore.similarity_search_with_score(query, k=10)
print("-" * 70)
for i, (doc, score) in enumerate(base_results):
    print(f"  [{i+1:2d}] L2 Distance: {score:.4f} | \"{doc.page_content[:80]}...\"")

# Cross-encoder reranking
print(f"\n[STEP 2] Cross-Encoder reranks to top 3:")
print("-" * 70)
reranked = cross_encoder_retriever.invoke(query)
for i, doc in enumerate(reranked):
    score = doc.metadata.get("relevance_score", "N/A")
    print(f"  [{i+1}] Relevance Score: {score}")
    print(f"      \"{doc.page_content[:150]}...\"")
    print()

print("=" * 70)
print("The Cross-Encoder considers query-document interactions for better ranking!")
print("=" * 70)

INFO:sentence_transformers.base.model:No device provided, using cpu


Loading Cross-Encoder model...


INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 404 Not Found"
INFO:sentence_transformers.base.model:No modules.json found for cross-encoder/ms-marco-MiniLM-L-6-v2, initializing a new CrossEncoder model.
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L-6-v2 "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Reque

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/adapter_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cross-encoder/ms-marco-MiniLM-L6-v2/c5ee24cb16019beea0893ab7796b1df96625c6b8/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/model.safetensors "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-

...

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/chat_template.jinja "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"


Cross-Encoder model loaded!

CROSS-ENCODER RERANKER DEMONSTRATION
Query: "What is deep learning and how does it work?"

[STEP 1] Base retriever fetches top 10 chunks
----------------------------------------------------------------------
  [ 1] L2 Distance: 0.3291 | "Deep Learning is a subset of machine learning that uses artificial neural networ..."
  [ 2] L2 Distance: 0.5344 | "Section 3: Deep Learning and Neural Networks..."
  [ 3] L2 Distance: 0.5415 | ". Deep learning, particularly CNNs like ResNet and EfficientNet, has dramaticall..."
  [ 4] L2 Distance: 0.6203 | ". Key architectures include Convolutional Neural Networks (CNNs) which are desig..."
  [ 5] L2 Distance: 0.6679 | "Machine Learning (ML) is a subset of AI that enables systems to learn and improv..."
  [ 6] L2 Distance: 0.6782 | "AI is transforming healthcare through numerous applications. Machine learning mo..."
  [ 7] L2 Distance: 0.6971 | "Natural Language Processing (NLP) is a branch of AI focused on the interaction 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  [1] Relevance Score: N/A
      "Deep Learning is a subset of machine learning that uses artificial neural networks with multiple layers to model complex patterns. A neural network co..."

  [2] Relevance Score: N/A
      ". Deep learning, particularly CNNs like ResNet and EfficientNet, has dramatically improved computer vision accuracy since AlexNet won the ImageNet com..."

  [3] Relevance Score: N/A
      "Section 3: Deep Learning and Neural Networks..."

The Cross-Encoder considers query-document interactions for better ranking!


---
### **Reranker 3: BM25 + Hybrid Search (Ensemble Retriever)**
---

### What is BM25?

**BM25** (Best Matching 25) is a classic **keyword-based** ranking algorithm.  
Unlike embeddings (semantic search), BM25 uses **exact word matching** with TF-IDF scoring.

| Feature | Semantic Search (Embeddings) | Keyword Search (BM25) |
|---|---|---|
| Matching | Meaning-based | Word-based |
| Strengths | "dog" matches "puppy" | Exact terms, names, codes |
| Weaknesses | May miss exact keywords | Misses synonyms |

### Hybrid Search = Semantic + Keyword

The **EnsembleRetriever** combines multiple retrievers with configurable weights:

```
Query ──► Semantic Retriever (FAISS) ──┐
          weight: 0.5                   ├──► Merge & Deduplicate ──► Final Results
Query ──► Keyword Retriever (BM25)  ──┘
          weight: 0.5
```

This gives you the **best of both worlds**!

In [24]:
# ============================================================
# BM25 + HYBRID SEARCH (EnsembleRetriever)
# ============================================================

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# BM25 Retriever (keyword-based)
bm25_retriever = BM25Retriever.from_documents(chunks, k=4)

# FAISS Retriever (semantic/embedding-based)
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Ensemble = Hybrid Retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[faiss_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

query = "What is transfer learning and LoRA fine-tuning?"

print("HYBRID SEARCH DEMONSTRATION")
print("=" * 70)
print(f"Query: \"{query}\"")

print(f"\n--- SEMANTIC SEARCH (FAISS) Results ---")
semantic_results = faiss_retriever.invoke(query)
for i, doc in enumerate(semantic_results):
    print(f"  [{i+1}] \"{doc.page_content[:120]}...\"")

print(f"\n--- KEYWORD SEARCH (BM25) Results ---")
bm25_results = bm25_retriever.invoke(query)
for i, doc in enumerate(bm25_results):
    print(f"  [{i+1}] \"{doc.page_content[:120]}...\"")

print(f"\n--- HYBRID SEARCH (Ensemble) Results ---")
hybrid_results = ensemble_retriever.invoke(query)
for i, doc in enumerate(hybrid_results):
    print(f"  [{i+1}] \"{doc.page_content[:120]}...\"")

print(f"\n{'=' * 70}")
print(f"Semantic found {len(semantic_results)} chunks (meaning-based)")
print(f"BM25 found {len(bm25_results)} chunks (keyword-based)")
print(f"Hybrid combined: {len(hybrid_results)} unique chunks (best of both!)")
print(f"{'=' * 70}")

HYBRID SEARCH DEMONSTRATION
Query: "What is transfer learning and LoRA fine-tuning?"

--- SEMANTIC SEARCH (FAISS) Results ---
  [1] "Section 8: Transfer Learning and Fine-Tuning..."
  [2] "Transfer learning is a technique where a model trained on one task is repurposed for a different but related task. Inste..."
  [3] ". Popular approaches include full fine-tuning where all model parameters are updated, LoRA (Low-Rank Adaptation) which a..."
  [4] "Section 3: Deep Learning and Neural Networks..."

--- KEYWORD SEARCH (BM25) Results ---
  [1] "Section 1: What is Artificial Intelligence?..."
  [2] ". Popular approaches include full fine-tuning where all model parameters are updated, LoRA (Low-Rank Adaptation) which a..."
  [3] "Transfer learning is a technique where a model trained on one task is repurposed for a different but related task. Inste..."
  [4] "Artificial Intelligence (AI) is the simulation of human intelligence processes by computer systems. These processes incl..."

--- HYB

---
## **PART 11: Complete RAG Pipeline with Reranking**
---

Now let's put it all together — a **production-quality** RAG pipeline:

```
Query
  │
  ├──► Semantic Retriever (FAISS, k=10) ──┐
  │                                        ├──► Ensemble (k=10 unique)
  ├──► Keyword Retriever (BM25, k=10)  ──┘
  │                                        │
  │                                        ▼
  │                                   FlashRank Reranker
  │                                   (rerank → top 3)
  │                                        │
  │                                        ▼
  └──► RunnablePassthrough ──────────► Prompt Template
                                           │
                                           ▼
                                         LLM
                                           │
                                           ▼
                                     Final Answer
```

In [25]:
# ============================================================
# FULL RAG PIPELINE WITH RERANKING
# ============================================================

# Step 1: Create hybrid base retriever
bm25_ret = BM25Retriever.from_documents(chunks, k=10)
faiss_ret = vectorstore.as_retriever(search_kwargs={"k": 10})

hybrid_retriever = EnsembleRetriever(
    retrievers=[faiss_ret, bm25_ret],
    weights=[0.5, 0.5]
)

# Step 2: Add reranking on top
rerank_compressor = FlashrankRerank(top_n=3)
reranking_retriever = ContextualCompressionRetriever(
    base_compressor=rerank_compressor,
    base_retriever=hybrid_retriever
)

# Step 3: Setup LLM (in case this cell is run independently)
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline as hf_pipeline
from langchain_huggingface import HuggingFacePipeline

model_id = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
pipe = hf_pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=256, temperature=0.1)
llm = HuggingFacePipeline(pipeline=pipe)

# Step 4: Build the RAG chain
rag_template = """You are a helpful AI teaching assistant. Answer the question based ONLY on the provided context.
Give a detailed, educational answer. If the context doesn't contain enough information, say so.

Context:
{context}

Question: {question}

Detailed Answer:"""

rag_prompt = PromptTemplate.from_template(rag_template)

def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs):
        formatted.append(f"[Source {i+1}]: {doc.page_content}")
    return "\n\n".join(formatted)

full_rag_chain = (
    {"context": reranking_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("FULL RAG + RERANKING PIPELINE BUILT!")
print("=" * 60)
print("Pipeline components:")
print("  1. Hybrid Retriever (FAISS + BM25)")
print("  2. FlashRank Reranker (top 3)")
print("  3. Prompt Template")
print("  4. LLM (flan-t5-small)")
print("  5. Output Parser")
print("=" * 60)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/flan-t5-small/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-small/0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/flan-t5-small/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-small/0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google/flan-t5-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google/flan-t5-small/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/flan-t5-small/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-small/0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab/generation_config.json "HTTP/1.1 200 OK"
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNe

FULL RAG + RERANKING PIPELINE BUILT!
Pipeline components:
  1. Hybrid Retriever (FAISS + BM25)
  2. FlashRank Reranker (top 3)
  3. Prompt Template
  4. LLM (flan-t5-small)
  5. Output Parser


In [26]:
# ============================================================
# RUN THE FULL PIPELINE — Detailed trace of every step
# ============================================================

query = "Explain how RAG works and why it uses vector databases"

print("━" * 70)
print("  FULL RAG + RERANKING PIPELINE EXECUTION")
print("━" * 70)
print(f"  Query: \"{query}\"")
print("━" * 70)

# Trace Step 1: Hybrid retrieval
print("\n[STEP 1] HYBRID RETRIEVAL")
print("-" * 70)
print("  Running FAISS (semantic) + BM25 (keyword) in parallel...")
hybrid_results = hybrid_retriever.invoke(query)
print(f"  Hybrid retriever returned {len(hybrid_results)} chunks")
for i, doc in enumerate(hybrid_results[:5]):
    print(f"    [{i+1}] \"{doc.page_content[:80]}...\"")
if len(hybrid_results) > 5:
    print(f"    ... and {len(hybrid_results) - 5} more chunks")

# Trace Step 2: Reranking
print("\n[STEP 2] RERANKING (FlashRank)")
print("-" * 70)
print(f"  Re-scoring {len(hybrid_results)} chunks and selecting top 3...")
reranked = reranking_retriever.invoke(query)
for i, doc in enumerate(reranked):
    score = doc.metadata.get("relevance_score", "N/A")
    print(f"    [{i+1}] Score: {score}")
    print(f"        \"{doc.page_content[:100]}...\"")

# Trace Step 3: Context formatting
print("\n[STEP 3] CONTEXT FORMATTING")
print("-" * 70)
context = format_docs(reranked)
print(f"  Combined context length: {len(context)} characters")

# Trace Step 4: Generate answer
print("\n[STEP 4] LLM GENERATION")
print("-" * 70)
answer = full_rag_chain.invoke(query)
print(f"\n{'█' * 70}")
print(f"  FINAL ANSWER:")
print(f"  {answer}")
print(f"{'█' * 70}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  FULL RAG + RERANKING PIPELINE EXECUTION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Query: "Explain how RAG works and why it uses vector databases"
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[STEP 1] HYBRID RETRIEVAL
----------------------------------------------------------------------
  Running FAISS (semantic) + BM25 (keyword) in parallel...
  Hybrid retriever returned 15 chunks
    [1] ". The RAG process involves three main steps: first, indexing where documents are..."
    [2] ". Vector databases like FAISS (Facebook AI Similarity Search), Pinecone, Chroma,..."
    [3] "Retrieval-Augmented Generation (RAG) is a technique that enhances large language..."
    [4] "Vector databases store data as high-dimensional vectors, enabling efficient simi..."
    [5] "Section 7: Vector Databases and Embeddings..."
    ... and 10 more chunks

[STEP 2] RERANKING (Flas

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



██████████████████████████████████████████████████████████████████████
  FINAL ANSWER:
  You are a helpful AI teaching assistant. Answer the question based ONLY on the provided context.
Give a detailed, educational answer. If the context doesn't contain enough information, say so.

Context:
[Source 1]: . Vector databases like FAISS (Facebook AI Similarity Search), Pinecone, Chroma, and Weaviate power modern search and RAG applications. FAISS is an open-source library that enables efficient similarity search in high-dimensional spaces.

[Source 2]: Vector databases store data as high-dimensional vectors, enabling efficient similarity search. Text embeddings transform words and sentences into dense numerical vectors that capture semantic meaning. When two texts discuss similar concepts, their embedding vectors will be close together in the vector space, even if they use completely different words. Popular embedding models include BGE (BAAI General Embedding), Sentence-BERT, and E5

[Sou

---
## **PART 12: Grand Comparison — All Retrieval Methods**
---

Let's compare all the retrieval methods side by side to see how reranking improves results.

In [27]:
# ============================================================
# GRAND COMPARISON: All Retrieval Methods
# ============================================================

queries = [
    "What is deep learning?",
    "How does RAG improve LLM responses?",
    "What are the ethical concerns in artificial intelligence?",
]

# Setup all retrievers
basic_ret = vectorstore.as_retriever(search_kwargs={"k": 3})

flashrank_comp = FlashrankRerank(top_n=3)
flashrank_ret = ContextualCompressionRetriever(
    base_compressor=flashrank_comp,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": 10})
)

cross_enc_comp = CrossEncoderReranker(model=cross_encoder_model, top_n=3)
cross_enc_ret = ContextualCompressionRetriever(
    base_compressor=cross_enc_comp,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": 10})
)

methods = {
    "Basic (FAISS top-3)": basic_ret,
    "FlashRank Reranked": flashrank_ret,
    "Cross-Encoder Reranked": cross_enc_ret,
    "Hybrid + FlashRank": reranking_retriever,
}

print("GRAND COMPARISON OF RETRIEVAL METHODS")
print("=" * 70)

for query in queries:
    print(f"\n{'━' * 70}")
    print(f"  QUERY: \"{query}\"")
    print(f"{'━' * 70}")

    for method_name, ret in methods.items():
        results = ret.invoke(query)
        print(f"\n  {method_name}:")
        for i, doc in enumerate(results[:3]):
            score_info = ""
            if "relevance_score" in doc.metadata:
                score_info = f" (score: {doc.metadata['relevance_score']:.4f})"
            print(f"    [{i+1}]{score_info} \"{doc.page_content[:90]}...\"")

print(f"\n{'=' * 70}")
print("KEY TAKEAWAYS:")
print("  1. Basic retrieval is fast but may miss the best chunks")
print("  2. FlashRank is ultra-fast and improves ordering")
print("  3. Cross-Encoder is most accurate but slower")
print("  4. Hybrid + Reranking gives the best overall results")
print(f"{'=' * 70}")

GRAND COMPARISON OF RETRIEVAL METHODS

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  QUERY: "What is deep learning?"
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Basic (FAISS top-3):
    [1] "Deep Learning is a subset of machine learning that uses artificial neural networks with mu..."
    [2] "Section 3: Deep Learning and Neural Networks..."
    [3] ". Deep learning, particularly CNNs like ResNet and EfficientNet, has dramatically improved..."

  FlashRank Reranked:
    [1] (score: 0.9991) "Section 1: What is Artificial Intelligence?..."
    [2] (score: 0.9990) "Deep Learning is a subset of machine learning that uses artificial neural networks with mu..."
    [3] (score: 0.9955) "Transfer learning is a technique where a model trained on one task is repurposed for a dif..."


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


  Cross-Encoder Reranked:
    [1] "Deep Learning is a subset of machine learning that uses artificial neural networks with mu..."
    [2] ". Deep learning, particularly CNNs like ResNet and EfficientNet, has dramatically improved..."
    [3] "Section 3: Deep Learning and Neural Networks..."

  Hybrid + FlashRank:
    [1] (score: 0.9993) "Section 1: What is Artificial Intelligence?..."
    [2] (score: 0.9991) "Deep Learning is a subset of machine learning that uses artificial neural networks with mu..."
    [3] (score: 0.9947) "Transfer learning is a technique where a model trained on one task is repurposed for a dif..."

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  QUERY: "How does RAG improve LLM responses?"
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Basic (FAISS top-3):
    [1] "Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by..."
    [2] ". The RAG process involves three main steps:

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


  Cross-Encoder Reranked:
    [1] "Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by..."
    [2] ". The RAG process involves three main steps: first, indexing where documents are split int..."
    [3] "Section 6: Retrieval-Augmented Generation (RAG)..."

  Hybrid + FlashRank:
    [1] (score: 0.9970) "Section 2: Machine Learning Fundamentals..."
    [2] (score: 0.9961) "Machine Learning (ML) is a subset of AI that enables systems to learn and improve from exp..."
    [3] (score: 0.9914) ". The RAG process involves three main steps: first, indexing where documents are split int..."

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  QUERY: "What are the ethical concerns in artificial intelligence?"
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Basic (FAISS top-3):
    [1] "AI ethics involves addressing the moral and social implications of AI systems. Key concern..."
    [2] "Section 9: AI Ethics a

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


  Cross-Encoder Reranked:
    [1] "AI ethics involves addressing the moral and social implications of AI systems. Key concern..."
    [2] "Artificial Intelligence (AI) is the simulation of human intelligence processes by computer..."
    [3] "Artificial Intelligence: A Comprehensive Overview..."

  Hybrid + FlashRank:
    [1] (score: 0.9990) "Artificial Intelligence (AI) is the simulation of human intelligence processes by computer..."
    [2] (score: 0.9981) "Section 1: What is Artificial Intelligence?..."
    [3] (score: 0.9947) "Machine Learning (ML) is a subset of AI that enables systems to learn and improve from exp..."

KEY TAKEAWAYS:
  1. Basic retrieval is fast but may miss the best chunks
  2. FlashRank is ultra-fast and improves ordering
  3. Cross-Encoder is most accurate but slower
  4. Hybrid + Reranking gives the best overall results


In [28]:
# ============================================================
# COMPARE RAG ANSWERS: Basic vs Reranked
# ============================================================

query = "What is RAG and why does it use vector databases?"

print("RAG ANSWER COMPARISON")
print("=" * 70)
print(f"Query: \"{query}\"")

# RAG with basic retriever
basic_chain = (
    {"context": basic_ret | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# RAG with FlashRank reranked retriever
reranked_chain = (
    {"context": flashrank_ret | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# RAG with full hybrid + reranking
full_chain = (
    {"context": reranking_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(f"\n--- Answer with BASIC retrieval ---")
basic_answer = basic_chain.invoke(query)
print(f"  {basic_answer}")

print(f"\n--- Answer with FLASHRANK reranking ---")
reranked_answer = reranked_chain.invoke(query)
print(f"  {reranked_answer}")

print(f"\n--- Answer with HYBRID + RERANKING (full pipeline) ---")
full_answer = full_chain.invoke(query)
print(f"  {full_answer}")

print(f"\n--- Answer from LLM ALONE (no RAG) ---")
direct_answer = llm.invoke(query)
print(f"  {direct_answer}")

print(f"\n{'=' * 70}")
print("Better retrieval → Better context → Better answers!")
print(f"{'=' * 70}")

RAG ANSWER COMPARISON
Query: "What is RAG and why does it use vector databases?"

--- Answer with BASIC retrieval ---


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  You are a helpful AI teaching assistant. Answer the question based ONLY on the provided context.
Give a detailed, educational answer. If the context doesn't contain enough information, say so.

Context:
[Source 1]: . The RAG process involves three main steps: first, indexing where documents are split into chunks, converted to embeddings, and stored in a vector database; second, retrieval where a user query is embedded and similar document chunks are found using similarity search; and third, generation where the retrieved context is combined with the query and fed to an LLM to produce an accurate answer.

[Source 2]: Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by retrieving relevant information from external knowledge bases before generating responses. RAG addresses key limitations of LLMs such as knowledge cutoff dates, hallucination, and lack of domain-specific knowledge

[Source 3]: Section 6: Retrieval-Augmented Generation (RAG)

Questio

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  You are a helpful AI teaching assistant. Answer the question based ONLY on the provided context.
Give a detailed, educational answer. If the context doesn't contain enough information, say so.

Context:
[Source 1]: . Vector databases like FAISS (Facebook AI Similarity Search), Pinecone, Chroma, and Weaviate power modern search and RAG applications. FAISS is an open-source library that enables efficient similarity search in high-dimensional spaces.

[Source 2]: Vector databases store data as high-dimensional vectors, enabling efficient similarity search. Text embeddings transform words and sentences into dense numerical vectors that capture semantic meaning. When two texts discuss similar concepts, their embedding vectors will be close together in the vector space, even if they use completely different words. Popular embedding models include BGE (BAAI General Embedding), Sentence-BERT, and E5

[Source 3]: . The RAG process involves three main steps: first, indexing where documents are

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  You are a helpful AI teaching assistant. Answer the question based ONLY on the provided context.
Give a detailed, educational answer. If the context doesn't contain enough information, say so.

Context:
[Source 1]: . Vector databases like FAISS (Facebook AI Similarity Search), Pinecone, Chroma, and Weaviate power modern search and RAG applications. FAISS is an open-source library that enables efficient similarity search in high-dimensional spaces.

[Source 2]: Transfer learning is a technique where a model trained on one task is repurposed for a different but related task. Instead of training from scratch, you start with a pre-trained model and adapt it to your specific use case. Fine-tuning involves taking a pre-trained model and continuing training on a smaller, task-specific dataset. This approach dramatically reduces the data and compute needed compared to training from scratch

[Source 3]: Vector databases store data as high-dimensional vectors, enabling efficient similarity sea

---
## **Summary & Key Takeaways**
---

### The Complete RAG Pipeline

```
┌────────────────────────────────────────────────────────────────────────────────┐
│                           COMPLETE RAG ARCHITECTURE                            │
│                                                                                │
│  ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐                 │
│  │ Documents │──►│ Chunking │──►│Embeddings│──►│  Vector  │                 │
│  │ (PDF,TXT) │    │ (Split)  │    │ (BGE)    │    │  Store   │                 │
│  └──────────┘    └──────────┘    └──────────┘    │ (FAISS)  │                 │
│                                                   └────┬─────┘                 │
│                                                        │                       │
│  ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌────▼─────┐                 │
│  │  Answer  │◄──│   LLM    │◄──│  Prompt  │◄──│ Reranker │                 │
│  │          │    │(Flan-T5) │    │ Template │    │(FlashRank│                 │
│  └──────────┘    └──────────┘    └──────────┘    └──────────┘                 │
│                                                                                │
└────────────────────────────────────────────────────────────────────────────────┘
```

### Concepts Covered

| # | Concept | What We Learned |
|---|---------|-----------------|
| 1 | **Chunking** | Split documents into overlapping pieces for retrieval |
| 2 | **Embeddings** | Convert text to 384-dim vectors capturing meaning |
| 3 | **Vector Store** | FAISS stores embeddings in-memory for fast search |
| 4 | **Retrieval** | Find top-K similar chunks via cosine/L2 distance |
| 5 | **Chains** | LCEL pipe operator connects components sequentially |
| 6 | **RunnablePassthrough** | Forwards input unchanged alongside transformations |
| 7 | **RAG Pipeline** | Retrieve context → Fill prompt → Generate answer |
| 8 | **FlashRank** | Ultra-fast, lightweight reranker (~4MB) |
| 9 | **Cross-Encoder** | Most accurate reranker (query+doc processed together) |
| 10 | **BM25 + Hybrid** | Combine keyword + semantic search for best coverage |
| 11 | **EnsembleRetriever** | Merge multiple retrievers with configurable weights |

### Models Used (All Open-Source!)

| Component | Model | Size |
|-----------|-------|------|
| Embeddings | `BAAI/bge-small-en-v1.5` | ~90MB |
| LLM | `google/flan-t5-small` | ~300MB |
| Reranker (FlashRank) | Built-in FlashRank model | ~4MB |
| Reranker (Cross-Encoder) | `cross-encoder/ms-marco-MiniLM-L-6-v2` | ~80MB |

### Production Tips

1. **Use larger models** for better accuracy (e.g., `bge-large`, `flan-t5-base`)
2. **Tune chunk size** based on your documents (250-1000 chars typical)
3. **Adjust overlap** to prevent context loss at boundaries
4. **Retrieve more, rerank fewer** (e.g., retrieve 20, rerank to 3-5)
5. **Use persistent vector stores** (Pinecone, Chroma, Weaviate) in production
6. **Combine retrievers** (hybrid search) for best coverage

---
## **References & Further Reading**
---

### Documentation
- [LangChain Retrievers](https://docs.langchain.com/oss/python/integrations/retrievers)
- [LangChain Embeddings](https://docs.langchain.com/oss/python/integrations/embeddings)
- [FlashRank Reranker](https://docs.langchain.com/oss/python/integrations/retrievers/flashrank-reranker)
- [BGE on HuggingFace](https://docs.langchain.com/oss/python/integrations/embeddings/bge_huggingface)
- [FAISS by Meta](https://github.com/facebookresearch/faiss)

### Research & Articles
- [Mastering Reranking in RAG](https://medium.com/@abheshith7/mastering-reranking-in-rag-from-basic-retrieval-to-advanced-methods-db297530361a)
- [The Critical Role of Rerankers in RAG](https://medium.com/@akanshak/the-critical-role-of-rerankers-in-rag-98309f52abe5)
- [Hybrid Search and BM25](https://medium.com/ai-insights-cobet/hybrid-search-and-bm-25-advanced-retrieval-with-code-8cc9801fa454)
- [Re-Ranking Mechanisms in RAG Pipelines](https://medium.com/@adnanmasood/re-ranking-mechanisms-in-retrieval-augmented-generation-pipelines-an-overview-8e24303ee789)
- [Enhancing RAG Systems Using Reranking with LangChain](https://medium.com/@myscale/enhancing-advanced-rag-systems-using-reranking-with-langchain-523a0b840311)

### Models
- [BAAI/bge-small-en-v1.5](https://huggingface.co/BAAI/bge-small-en-v1.5) — Embedding model
- [google/flan-t5-small](https://huggingface.co/google/flan-t5-small) — LLM
- [cross-encoder/ms-marco-MiniLM-L-6-v2](https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2) — Cross-encoder reranker
- [FlashRank](https://github.com/PrithivirajDamodaran/FlashRank) — Lightweight reranker

---

**Congratulations!** You now have a complete understanding of RAG pipelines and reranking techniques.  
Feel free to experiment with different models, chunk sizes, and rerankers!

---
## **BONUS: End-to-End RAG + Reranker — Clean Production Code**
---

Copy-paste ready. No prints, no explanations — just straight working code.

In [27]:
!pip install -q langchain langchain-core langchain-classic langchain-community langchain-huggingface langchain-text-splitters sentence-transformers faiss-cpu flashrank transformers torch rank_bm25

import langchain, warnings
for attr in ("debug", "verbose", "llm_cache"):
    if not hasattr(langchain, attr):
        setattr(langchain, attr, False)
warnings.filterwarnings("ignore")

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_community.document_compressors import FlashrankRerank
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers import EnsembleRetriever
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

# ── 1. DOCUMENTS ──────────────────────────────────────────────
documents = [
    Document(page_content="Retrieval-Augmented Generation (RAG) enhances LLMs by retrieving relevant context from external knowledge bases before generating answers. It solves hallucination, knowledge cutoff, and domain gaps."),
    Document(page_content="Vector databases like FAISS, Pinecone, and Chroma store text as high-dimensional embedding vectors for fast similarity search. FAISS is open-source by Meta and runs in-memory."),
    Document(page_content="Text embeddings convert sentences into dense numerical vectors that capture semantic meaning. Models like BGE, Sentence-BERT, and E5 produce vectors where similar texts are close together."),
    Document(page_content="Rerankers re-score retrieved documents by processing query and document together (cross-encoder), unlike retrievers which embed them separately (bi-encoder). This gives higher accuracy."),
    Document(page_content="BM25 is a keyword-based ranking algorithm using term frequency and inverse document frequency. Hybrid search combines BM25 with semantic search for better recall."),
    Document(page_content="Deep learning uses multi-layer neural networks. CNNs handle images, RNNs handle sequences, and Transformers use self-attention for parallel processing of text."),
    Document(page_content="Transfer learning reuses a pre-trained model for a new task. Fine-tuning adapts it on task-specific data. LoRA adds small trainable matrices to reduce compute cost."),
    Document(page_content="AI ethics covers bias, transparency, privacy, job displacement, and environmental impact. Responsible AI requires diverse teams, bias testing, and tools like LIME and SHAP."),
]

# ── 2. CHUNK ──────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(documents)

# ── 3. EMBED + VECTOR STORE ──────────────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
vectorstore = FAISS.from_documents(chunks, embeddings)

# ── 4. RETRIEVERS ─────────────────────────────────────────────
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 6})
bm25_retriever = BM25Retriever.from_documents(chunks, k=6)

hybrid_retriever = EnsembleRetriever(
    retrievers=[faiss_retriever, bm25_retriever],
    weights=[0.5, 0.5],
)

# ── 5. RERANKER ───────────────────────────────────────────────
reranker = FlashrankRerank(top_n=3)
retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=hybrid_retriever,
)

# ── 6. LLM ────────────────────────────────────────────────────
model_id = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=256)
llm = HuggingFacePipeline(pipeline=pipe)

# ── 7. RAG CHAIN ──────────────────────────────────────────────
prompt = PromptTemplate.from_template(
    "Answer based on the context below.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)

rag_chain = (
    {"context": retriever | (lambda docs: "\n\n".join(d.page_content for d in docs)), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# ── 8. RUN ────────────────────────────────────────────────────
rag_chain.invoke("What is RAG and how does reranking improve it?")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5


INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-small/0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab/generation_config.json "HTTP/1.1 200 OK"


Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'ExaoneMoeForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalL

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Answer based on the context below.\n\nContext:\nTransfer learning reuses a pre-trained model for a new task. Fine-tuning adapts it on task-specific data. LoRA adds small trainable matrices to reduce compute cost.\n\nRetrieval-Augmented Generation (RAG) enhances LLMs by retrieving relevant context from external knowledge bases before generating answers. It solves hallucination, knowledge cutoff, and domain gaps.\n\nRerankers re-score retrieved documents by processing query and document together (cross-encoder), unlike retrievers which embed them separately (bi-encoder). This gives higher accuracy.\n\nQuestion: What is RAG and how does reranking improve it?\n\nAnswer:'